## 02: Preprocessing
Merge phenotype + environment + genomic data into one flat feature matrix ready for modeling.

Flip `SAMPLE_MODE = False` once ready to run on full data.

In [1]:
import pandas as pd
import numpy as np
import os
import re

SAMPLE_MODE = False  #flip to False for full run

#paths
if SAMPLE_MODE:
    PHENO_PATH = '../data/raw/sample_data/sample_C1_phenotype_100rows.csv'
    ENV_PATH   = '../data/raw/sample_data/sample_environmental_20rows.csv'
    def geno_path(cluster_id, pop_num):
        return f'../data/raw/sample_data/sample_C{cluster_id}.{pop_num}_100rows.csv'
else:
    PHENO_PATH = '../data/raw/C1_Phenotype_Data_V2.csv'
    ENV_PATH   = '../data/raw/environmental_features.csv'
    def geno_path(cluster_id, pop_num):
        return f'../data/raw/genotypes/C{cluster_id}/ImputedPopulationsC{cluster_id}/C{cluster_id}.{pop_num}_Imputed.csv'

OUTPUT_PATH = '../data/processed/merged_sample.csv' if SAMPLE_MODE else '../data/processed/merged_full.csv'
print('SAMPLE_MODE:', SAMPLE_MODE)


SAMPLE_MODE: False


### 1. Load raw data

In [2]:
pheno = pd.read_csv(PHENO_PATH)
env   = pd.read_csv(ENV_PATH)
print('Raw pheno:', pheno.shape)
print('Env:      ', env.shape)


Raw pheno: (536936, 33)
Env:       (1185, 86)


/tmp/job.346229/ipykernel_223872/4244469098.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  pheno = pd.read_csv(PHENO_PATH)


### 2. Clean phenotype
Drop junk columns left over from the pre-merge, normalize the YEAR column name, and parse `LINE_UNIQUE_ID` into its three components.

In [3]:
JUNK_COLS = [
    'Unnamed: 0', 'Unnamed: 0_x', 'Unnamed: 0_y',
    'projects_x', 'projects_y',
    'FILE_LIST', 'shorthand_x', 'shorthand_y',
    'YEAR_y', 'projectID', 'MAB_PROJECT_ID', 'GENERATION_NAME',
]
pheno = pheno.drop(columns=[c for c in JUNK_COLS if c in pheno.columns])

# normalize YEAR (sample pheno has YEAR_x due to pre-merge duplication)
if 'YEAR_x' in pheno.columns:
    pheno = pheno.rename(columns={'YEAR_x': 'YEAR'})

# parse LINE_UNIQUE_ID: 'C1.1.191' -> cluster=1, pop=1, line=191
parsed = pheno['LINE_UNIQUE_ID'].str.extract(r'^C(\d+)\.(\d+)\.(\d+)$')
parsed.columns = ['CLUSTER_ID', 'POP_NUM', 'LINE_NUM']
pheno[['CLUSTER_ID', 'POP_NUM', 'LINE_NUM']] = parsed

n_unparsed = pheno['LINE_UNIQUE_ID'][parsed['LINE_NUM'].isnull()].nunique()
print(f"Dropping {parsed['LINE_NUM'].isnull().sum()} rows with unparseable LINE_UNIQUE_ID ({n_unparsed} unique IDs)")

pheno = pheno[parsed['LINE_NUM'].notna()].copy()
pheno['CLUSTER_ID'] = pheno['CLUSTER_ID'].astype(int)
pheno['POP_NUM']    = pheno['POP_NUM'].astype(int)
pheno['LINE_NUM']   = pheno['LINE_NUM'].astype(int)
print(f'Clean pheno: {pheno.shape}  |  LINE_UNIQUE_ID parse failures: {n_unparsed}')
pheno[['LINE_UNIQUE_ID', 'CLUSTER_ID', 'POP_NUM', 'LINE_NUM']].head(3)


Dropping 4488 rows with unparseable LINE_UNIQUE_ID (693 unique IDs)
Clean pheno: (532448, 24)  |  LINE_UNIQUE_ID parse failures: 693


,LINE_UNIQUE_ID,CLUSTER_ID,POP_NUM,LINE_NUM
0,C1.1.191,1,1,191
1,C1.1.193,1,1,193
2,C1.1.62,1,1,62


### 3. Merge phenotype + environment
Left join on `YEAR + LOC` so every phenotype row keeps its env features (or gets NaN if the env row is missing — expected in sample mode).

In [4]:
pheno_env = pheno.merge(env, on=['YEAR', 'LOC'], how='left')
print(f'Pheno+Env shape: {pheno_env.shape}')

env_feat_cols = [c for c in env.columns if c not in ('YEAR', 'LOC')]
rows_no_env = pheno_env[env_feat_cols].isnull().all(axis=1).sum()
print(f'Rows with no env match: {rows_no_env}/{len(pheno_env)} (expected ~all in sample mode)')


Pheno+Env shape: (532448, 108)
Rows with no env match: 0/532448 (expected ~all in sample mode)


### 4. Merge with genomic data
For each (cluster, population) group in the phenotype, load the matching genomic file, filter to progeny rows only (dropping parent PIDs), and left-join on `LINE_NUM`.

In [10]:
def load_geno(cluster_id, pop_num):
    path = geno_path(cluster_id, pop_num)
    if not os.path.exists(path):
        print(f'  [MISS] {path}')
        return None
    geno = pd.read_csv(path, index_col=0)
    progeny_mask = geno.index.astype(str).str.match(r'^\d{11}$')
    geno = geno.loc[progeny_mask].copy()
    geno['LINE_NUM'] = geno.index.astype(int)
    return geno

groups = pheno_env.groupby(['CLUSTER_ID', 'POP_NUM'], sort=False)
print(f'Populations to process: {groups.ngroups}')

first_chunk = True
for (cluster_id, pop_num), group in groups:
    geno = load_geno(cluster_id, int(pop_num))
    if geno is not None:
        merged = group.merge(
            geno.drop(columns=['LINE_NUM']).assign(LINE_NUM=geno['LINE_NUM']),
            on='LINE_NUM', how='left')
        if len(group) and merged.shape[1] > group.shape[1]:
            print(f'  C{cluster_id}.{pop_num}: {len(group)} rows -> {merged.shape[1]} cols')
    else:
        merged = group.copy()
    
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    merged.to_csv(OUTPUT_PATH, mode='w' if first_chunk else 'a',
                  header=first_chunk, index=False)
    first_chunk = False

print(f'\nDone. Saved -> {OUTPUT_PATH}')

Populations to process: 495
  C1.1: 1070 rows -> 3019 cols
  C1.2: 795 rows -> 3019 cols
  C1.3: 1458 rows -> 3019 cols
  C1.4: 1034 rows -> 3019 cols
  C1.5: 666 rows -> 3019 cols
  C1.6: 843 rows -> 3019 cols
  C1.7: 808 rows -> 3019 cols
  C1.8: 1190 rows -> 3019 cols
  C1.9: 816 rows -> 3019 cols
  C1.10: 1032 rows -> 3019 cols
  C1.11: 811 rows -> 3019 cols
  C1.12: 1223 rows -> 3019 cols
  C1.13: 1116 rows -> 3019 cols
  C1.14: 1198 rows -> 3019 cols
  C1.15: 1008 rows -> 3019 cols
  C1.16: 787 rows -> 3019 cols
  C1.17: 680 rows -> 3019 cols
  C1.18: 819 rows -> 3019 cols
  C1.19: 819 rows -> 3019 cols
  C1.20: 1190 rows -> 3019 cols
  C1.21: 1148 rows -> 3019 cols
  C1.22: 960 rows -> 3019 cols
  C1.23: 930 rows -> 3019 cols
  C1.24: 1329 rows -> 3019 cols
  C1.25: 1285 rows -> 3019 cols
  C1.26: 1352 rows -> 3019 cols
  C1.27: 1362 rows -> 3019 cols
  C1.28: 1389 rows -> 3019 cols
  C1.29: 1239 rows -> 3019 cols
  C1.30: 1231 rows -> 3019 cols
  C1.31: 1502 rows -> 3019 cols
 

/tmp/job.346229/ipykernel_223872/3685475672.py:6: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  geno = pd.read_csv(path, index_col=0)


  C1.107: 2728 rows -> 3019 cols
  C1.108: 1381 rows -> 3019 cols
  C1.109: 1335 rows -> 3019 cols
  C1.110: 1449 rows -> 3019 cols
  C1.111: 1395 rows -> 3019 cols
  C1.112: 1228 rows -> 3019 cols
  C1.113: 1447 rows -> 3019 cols
  C1.114: 1050 rows -> 3019 cols
  C1.115: 1380 rows -> 3019 cols
  C1.116: 1243 rows -> 3019 cols
  C1.117: 1430 rows -> 3019 cols
  C1.118: 1042 rows -> 3019 cols
  C1.119: 1220 rows -> 3019 cols
  C1.120: 1260 rows -> 3019 cols
  C1.121: 1471 rows -> 3019 cols
  C1.122: 1219 rows -> 3019 cols
  C1.123: 1480 rows -> 3019 cols
  C1.124: 719 rows -> 3019 cols
  C1.127: 736 rows -> 3019 cols
  C1.128: 1408 rows -> 3019 cols
  C1.129: 1466 rows -> 3019 cols
  C1.130: 1440 rows -> 3019 cols
  C1.131: 1258 rows -> 3019 cols
  C1.132: 1218 rows -> 3019 cols
  C1.133: 1432 rows -> 3019 cols
  C1.134: 1195 rows -> 3019 cols
  C1.135: 1077 rows -> 3019 cols
  C1.136: 1260 rows -> 3019 cols
  C1.137: 1440 rows -> 3019 cols
  C1.138: 1245 rows -> 3019 cols
  C1.139: 54

/tmp/job.346229/ipykernel_223872/3685475672.py:6: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  geno = pd.read_csv(path, index_col=0)


  C1.237: 2728 rows -> 3019 cols
  C1.238: 1790 rows -> 3019 cols
  C1.239: 1435 rows -> 3019 cols
  C1.240: 1448 rows -> 3019 cols
  C1.241: 1448 rows -> 3019 cols
  C1.242: 1457 rows -> 3019 cols
  C1.243: 1425 rows -> 3019 cols
  C1.244: 1462 rows -> 3019 cols
  C1.245: 1439 rows -> 3019 cols
  C1.246: 1480 rows -> 3019 cols
  C1.247: 1471 rows -> 3019 cols
  C1.248: 1456 rows -> 3019 cols
  C1.249: 1237 rows -> 3019 cols
  C1.250: 1281 rows -> 3019 cols
  C1.251: 1611 rows -> 3019 cols
  C1.252: 1406 rows -> 3019 cols
  C1.253: 1260 rows -> 3019 cols
  C1.254: 1312 rows -> 3019 cols
  C1.255: 1416 rows -> 3019 cols
  C1.256: 1392 rows -> 3019 cols
  C1.257: 1464 rows -> 3019 cols
  C1.258: 1440 rows -> 3019 cols
  C1.259: 2155 rows -> 3019 cols
  C1.260: 1436 rows -> 3019 cols
  C1.261: 1440 rows -> 3019 cols
  C1.262: 1063 rows -> 3019 cols
  C1.263: 1423 rows -> 3019 cols
  C1.264: 1438 rows -> 3019 cols
  C1.265: 1431 rows -> 3019 cols
  C1.266: 1252 rows -> 3019 cols
  C1.267: 

  C1.491: 495 rows -> 3019 cols
  C1.492: 838 rows -> 3019 cols
  C1.493: 583 rows -> 3019 cols
  C1.494: 255 rows -> 3019 cols
  C1.495: 1247 rows -> 3019 cols
  C1.496: 1444 rows -> 3019 cols
  C1.497: 1234 rows -> 3019 cols
  C1.498: 458 rows -> 3019 cols
  C1.499: 914 rows -> 3019 cols
  C1.500: 929 rows -> 3019 cols

Done. Saved -> ../data/processed/merged_full.csv


### 5. Quality check

In [7]:
# Target variable
print(f'YLD_BE missing: {df["YLD_BE"].isnull().sum()}/{len(df)}')

# SNP missingness (pre-imputed files should be ~0%)
snp_cols = [c for c in df.columns if c.startswith('M')]
if snp_cols:
    snp_null_pct = df[snp_cols].isnull().mean().mean() * 100
    print(f'SNP cols: {len(snp_cols)}  |  mean missingness: {snp_null_pct:.1f}%')
else:
    print('No SNP columns merged (check geno file match)')

# Env missingness
env_null_pct = df[env_feat_cols].isnull().mean().mean() * 100
print(f'Env feat missingness: {env_null_pct:.1f}%')

# Preview
key_cols = ['LINE_UNIQUE_ID', 'YEAR', 'LOC', 'YLD_BE'] + snp_cols[:3] + env_feat_cols[:3]
df[[c for c in key_cols if c in df.columns]].head(3)


YLD_BE missing: 20806/532448
SNP cols: 2912  |  mean missingness: 14.1%
Env feat missingness: 0.0%


,LINE_UNIQUE_ID,YEAR,LOC,YLD_BE,MST,M00003409443,M00000005000,X04_PRCP,X05_PRCP,X06_PRCP
0,C1.1.191,2001,NEDA,156.052,19.7,1.0,1.0,132.4,149.2,65.4
1,C1.1.193,2001,NEDA,150.685,20.2,1.0,1.0,132.4,149.2,65.4
2,C1.1.62,2001,IAPR,157.218,19.1,1.0,1.0,136.9,158.3,51.5


### 6. Save

SAMPLE DATA INTERP:

The two 100% missingness numbers are both sample artifacts. The env miss at 100% is the same LOC mismatch seen in the 01_eda file. The SNP miss at 65.5% comes directly from our EDA finding that the sample geno file covers lines 1-100, but pheno lines mostly run 62-198, so only line 62 and a handful of low-numbered lines matched. Every unmatched row gets NaN across all 2912 SNP columns. That math lands us right around 60-65% missingness. Row 2 on line 62 shows actual SNP values (1.0, 1.0) while rows 0-1 (lines 191, 193) are all NaN.

When we flip to SAMPLE_MODE = False:

Env missingness should drop to near 0% (full env file covers all LOC+YEAR combos)
SNP missingness should drop to near 0% (full geno files are pre-imputed, and every pheno line will have a matching geno file)
The geno loop will spin over hundreds of populations instead of 1, so expect a few minutes of runtime

The pipeline is structurally correct & The 3/100 missing YLD_BE is the only real data gap, and those 3 rows need to be dropped before modeling since that's the target.